# Structural Health Assessment of an RCC Beam Using Fuzzy Logic

**Subject:** Fuzzy Logic and Genetic Programming
**GTU Subject Code:** BE05000461
**Milestone 2 / Phase 3 – Python Implementation**

This notebook implements a fuzzy logic based system to estimate the structural
health condition of an RCC (Reinforced Cement Concrete) beam using two field
measurable parameters: **crack width** and **deflection**.

## Objective

To build a fuzzy inference system that takes crack width and deflection of an
RCC beam as inputs and estimates a **Structural Health Index**, which tells us
whether the beam condition is Safe, Moderate (needs monitoring) or Critical
(needs attention). Triangular membership functions are used throughout, and
the final result is calculated using the **weighted average defuzzification
method**, as required for this milestone.

## Step 1: Install Required Libraries

We need `scikit-fuzzy` for the fuzzy logic toolbox, along with `numpy` and
`matplotlib` for calculations and graphs.

In [ ]:
!pip install scikit-fuzzy matplotlib numpy

## Step 2: Import Libraries

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import pandas as pd

## Step 3: Define Input and Output Variables

We are using two inputs and one output:

- **Crack Width** (mm) – ranges from 0 to 6 mm
- **Deflection** (mm) – ranges from 0 to 25 mm
- **Structural Health Index** (0 to 100) – 0 means critical/poor condition,
  100 means safe/good condition

These ranges are taken from the System Design prepared in Milestone 1.

In [ ]:
# Antecedents (inputs)
crack = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'crack_width')
deflection = ctrl.Antecedent(np.arange(0, 25.01, 0.01), 'deflection')

# Consequent (output)
health = ctrl.Consequent(np.arange(0, 100.01, 0.01), 'health_index')

## Step 4: Define Crack Width Membership Functions

Only **triangular** membership functions (`fuzz.trimf`) are used, as required
for this project. The parameters below match the System Design document.

In [ ]:
crack['low']    = fuzz.trimf(crack.universe, [0, 0, 1.5])
crack['medium'] = fuzz.trimf(crack.universe, [0.5, 2, 3.5])
crack['high']   = fuzz.trimf(crack.universe, [2.5, 6, 6])

## Step 5: Define Deflection Membership Functions

In [ ]:
deflection['low']    = fuzz.trimf(deflection.universe, [0, 0, 8])
deflection['medium'] = fuzz.trimf(deflection.universe, [5, 12.5, 20])
deflection['high']   = fuzz.trimf(deflection.universe, [15, 25, 25])

## Step 6: Define Structural Health Index Membership Functions

`critical` represents a poor beam condition, `moderate` represents a warning
condition that needs monitoring, and `safe` represents a good condition.

In [ ]:
health['critical'] = fuzz.trimf(health.universe, [0, 0, 40])
health['moderate'] = fuzz.trimf(health.universe, [20, 50, 80])
health['safe']     = fuzz.trimf(health.universe, [60, 100, 100])

## Step 7: Display Membership Function Graphs

Here we plot all three membership functions to visually check that they are
defined correctly.

In [ ]:
crack.view()
plt.title('Crack Width Membership Functions')
plt.show()

deflection.view()
plt.title('Deflection Membership Functions')
plt.show()

health.view()
plt.title('Structural Health Index Membership Functions')
plt.show()

## Step 8: Create Fuzzy Rule Base

The following 9 rules are used to estimate the structural health of the beam,
based on standard RCC beam condition assessment logic:

| Rule | Crack Width | Deflection | Structural Health |
|------|-------------|------------|--------------------|
| 1 | Low    | Low    | Safe |
| 2 | Low    | Medium | Safe |
| 3 | Low    | High   | Moderate |
| 4 | Medium | Low    | Moderate |
| 5 | Medium | Medium | Moderate |
| 6 | Medium | High   | Critical |
| 7 | High   | Low    | Moderate |
| 8 | High   | Medium | Critical |
| 9 | High   | High   | Critical |

In [ ]:
rule1 = ctrl.Rule(crack['low'] & deflection['low'], health['safe'])
rule2 = ctrl.Rule(crack['low'] & deflection['medium'], health['safe'])
rule3 = ctrl.Rule(crack['low'] & deflection['high'], health['moderate'])
rule4 = ctrl.Rule(crack['medium'] & deflection['low'], health['moderate'])
rule5 = ctrl.Rule(crack['medium'] & deflection['medium'], health['moderate'])
rule6 = ctrl.Rule(crack['medium'] & deflection['high'], health['critical'])
rule7 = ctrl.Rule(crack['high'] & deflection['low'], health['moderate'])
rule8 = ctrl.Rule(crack['high'] & deflection['medium'], health['critical'])
rule9 = ctrl.Rule(crack['high'] & deflection['high'], health['critical'])

rules = [rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9]

## Step 9: Create Fuzzy Controller

Now we build the control system and the simulation object using all 9 rules.

In [ ]:
health_ctrl = ctrl.ControlSystem(rules)
system = ctrl.ControlSystemSimulation(health_ctrl)

## Step 10: Give Sample RCC Beam Data

Now we provide a sample crack width and deflection value to test the system.
This is an academic sample case, not real site data.

- Crack Width = **1.0 mm**
- Deflection = **17.0 mm**

These values were chosen so that more than one rule fires, which helps in
demonstrating the weighted average calculation clearly.

In [ ]:
sample_crack = 1.0       # mm
sample_deflection = 17.0  # mm

print("Sample Crack Width  :", sample_crack, "mm")
print("Sample Deflection   :", sample_deflection, "mm")

## Step 11: Calculate Membership Values

Here we calculate how much the sample input belongs to each linguistic
category, using `fuzz.interp_membership()`.

In [ ]:
crack_low_val    = fuzz.interp_membership(crack.universe, crack['low'].mf, sample_crack)
crack_med_val    = fuzz.interp_membership(crack.universe, crack['medium'].mf, sample_crack)
crack_high_val   = fuzz.interp_membership(crack.universe, crack['high'].mf, sample_crack)

defl_low_val     = fuzz.interp_membership(deflection.universe, deflection['low'].mf, sample_deflection)
defl_med_val     = fuzz.interp_membership(deflection.universe, deflection['medium'].mf, sample_deflection)
defl_high_val    = fuzz.interp_membership(deflection.universe, deflection['high'].mf, sample_deflection)

print("Crack Width Membership:")
print(f"  Low    = {crack_low_val:.4f}")
print(f"  Medium = {crack_med_val:.4f}")
print(f"  High   = {crack_high_val:.4f}")

print("\nDeflection Membership:")
print(f"  Low    = {defl_low_val:.4f}")
print(f"  Medium = {defl_med_val:.4f}")
print(f"  High   = {defl_high_val:.4f}")

## Step 12: Identify Fired Rules and Calculate Firing Strength

For AND conditions, the firing strength of a rule is the **minimum** of the
input membership values used in that rule. A rule is considered "fired" only
if its firing strength is greater than zero. Each output category (safe,
moderate, critical) is also assigned a representative crisp value, used later
for the weighted average calculation.

In [ ]:
# Representative crisp value for each output category (used in weighted average)
crisp_value = {'safe': 85, 'moderate': 50, 'critical': 15}

# All 9 rules with their input membership values and output category
rule_table = [
    (1, 'low',    'low',    crack_low_val,  defl_low_val,  'safe'),
    (2, 'low',    'medium', crack_low_val,  defl_med_val,  'safe'),
    (3, 'low',    'high',   crack_low_val,  defl_high_val, 'moderate'),
    (4, 'medium', 'low',    crack_med_val,  defl_low_val,  'moderate'),
    (5, 'medium', 'medium', crack_med_val,  defl_med_val,  'moderate'),
    (6, 'medium', 'high',   crack_med_val,  defl_high_val, 'critical'),
    (7, 'high',   'low',    crack_high_val, defl_low_val,  'moderate'),
    (8, 'high',   'medium', crack_high_val, defl_med_val,  'critical'),
    (9, 'high',   'high',   crack_high_val, defl_high_val, 'critical'),
]

fired_rules = []
for rule_no, crack_term, defl_term, crack_val, defl_val, output_term in rule_table:
    strength = min(crack_val, defl_val)   # AND -> minimum
    if strength > 0:
        fired_rules.append({
            'Rule': rule_no,
            'Crack Width': crack_term,
            'Deflection': defl_term,
            'Firing Strength': round(strength, 4),
            'Output Category': output_term,
            'Output Crisp Value': crisp_value[output_term]
        })

fired_df = pd.DataFrame(fired_rules)
fired_df

## Step 13: Weighted Average Defuzzification

As required for this milestone, the final crisp output is calculated
explicitly using the weighted average formula:

**Crisp Output = Σ(wᵢ × zᵢ) / Σ(wᵢ)**

where `wᵢ` is the firing strength of a fired rule and `zᵢ` is the
representative crisp output value of that rule's consequent.

Note: this manual weighted-average calculation is kept separate from
scikit-fuzzy's built-in `ControlSystemSimulation`, which internally uses the
centroid method. Both are shown below for comparison.

In [ ]:
weighted_sum = sum(r['Firing Strength'] * r['Output Crisp Value'] for r in fired_rules)
total_strength = sum(r['Firing Strength'] for r in fired_rules)
final_health_index = weighted_sum / total_strength

print("Weighted Sum            =", round(weighted_sum, 4))
print("Total Firing Strength   =", round(total_strength, 4))
print("Final Structural Health Index (Weighted Average) =", round(final_health_index, 2))

## Step 14: Compare With scikit-fuzzy's Built-in Centroid Result

Just to see the difference, we also run the sample through scikit-fuzzy's
own `ControlSystemSimulation`, which uses the **centroid** method by default.
This value is shown only for comparison — our official project result is the
weighted average value calculated above.

In [ ]:
system.input['crack_width'] = sample_crack
system.input['deflection'] = sample_deflection
system.compute()

centroid_result = system.output['health_index']
print("scikit-fuzzy centroid-based result :", round(centroid_result, 2))
print("Our weighted-average result        :", round(final_health_index, 2))

## Step 15: Health Condition Interpretation

Based on the Structural Health Index scale used in this project:

- **0 – 40** → Critical / Poor condition
- **20 – 80** → Moderate / Warning condition (needs monitoring)
- **60 – 100** → Safe / Good condition

We classify the final result accordingly.

In [ ]:
def classify_health(index_value):
    if index_value <= 35:
        return "Critical / Poor"
    elif index_value <= 65:
        return "Moderate / Warning"
    else:
        return "Safe / Good"

condition = classify_health(final_health_index)
print(f"Final Structural Health Index : {round(final_health_index, 2)}")
print(f"Beam Condition                : {condition}")

**Note:** This fuzzy logic model is an academic demonstration. Actual RCC
beam structural safety assessment requires proper site inspection, material
testing, structural analysis and evaluation by a qualified structural
engineer.

## Step 16: Defuzzification / Result Graph

Here we visualize the Structural Health Index membership functions along with
the final crisp output (from weighted average) marked on the graph.

In [ ]:
health.view(sim=system)
plt.axvline(final_health_index, color='green', linestyle='--',
            label=f'Weighted Avg Result = {round(final_health_index,2)}')
plt.legend()
plt.title('Structural Health Index - Result')
plt.show()

## Step 17: Test Different RCC Beam Cases

Now we test the system with 5 different crack width and deflection
combinations, covering low, medium and high conditions, to see how the
Structural Health Index changes.

In [ ]:
def evaluate_beam(cw, defl):
    """Runs one crack width / deflection pair through the fuzzy system
    and returns the weighted-average Structural Health Index."""

    c_low  = fuzz.interp_membership(crack.universe, crack['low'].mf, cw)
    c_med  = fuzz.interp_membership(crack.universe, crack['medium'].mf, cw)
    c_high = fuzz.interp_membership(crack.universe, crack['high'].mf, cw)

    d_low  = fuzz.interp_membership(deflection.universe, deflection['low'].mf, defl)
    d_med  = fuzz.interp_membership(deflection.universe, deflection['medium'].mf, defl)
    d_high = fuzz.interp_membership(deflection.universe, deflection['high'].mf, defl)

    local_rule_table = [
        ('low', 'low', c_low, d_low, 'safe'),
        ('low', 'medium', c_low, d_med, 'safe'),
        ('low', 'high', c_low, d_high, 'moderate'),
        ('medium', 'low', c_med, d_low, 'moderate'),
        ('medium', 'medium', c_med, d_med, 'moderate'),
        ('medium', 'high', c_med, d_high, 'critical'),
        ('high', 'low', c_high, d_low, 'moderate'),
        ('high', 'medium', c_high, d_med, 'critical'),
        ('high', 'high', c_high, d_high, 'critical'),
    ]

    w_sum = 0.0
    w_total = 0.0
    for _, _, cval, dval, out_term in local_rule_table:
        strength = min(cval, dval)
        if strength > 0:
            w_sum += strength * crisp_value[out_term]
            w_total += strength

    result = w_sum / w_total if w_total > 0 else 0.0
    return round(result, 2), classify_health(result)


test_cases = [
    {"Case": "Test 1 - Low crack + Low deflection",    "Crack Width (mm)": 0.3, "Deflection (mm)": 3},
    {"Case": "Test 2 - Medium crack + Medium deflection", "Crack Width (mm)": 2.0, "Deflection (mm)": 12.5},
    {"Case": "Test 3 - High crack + Medium deflection", "Crack Width (mm)": 5.0, "Deflection (mm)": 12},
    {"Case": "Test 4 - High crack + High deflection",   "Crack Width (mm)": 5.5, "Deflection (mm)": 22},
    {"Case": "Test 5 - Mixed condition",                "Crack Width (mm)": 1.5, "Deflection (mm)": 15},
]

for tc in test_cases:
    idx, cond = evaluate_beam(tc["Crack Width (mm)"], tc["Deflection (mm)"])
    tc["Structural Health Index"] = idx
    tc["Beam Condition"] = cond

results_df = pd.DataFrame(test_cases)
results_df

## Step 18: Test Case Result Comparison

A simple bar chart to compare the Structural Health Index across all test
cases.

In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.bar(results_df["Case"], results_df["Structural Health Index"],
               color=['green' if v > 65 else 'orange' if v > 35 else 'red'
                      for v in results_df["Structural Health Index"]])
plt.ylabel("Structural Health Index")
plt.xticks(rotation=30, ha='right')
plt.title("Structural Health Index - Test Cases")
plt.axhline(65, color='green', linestyle=':', linewidth=1)
plt.axhline(35, color='red', linestyle=':', linewidth=1)
plt.tight_layout()
plt.show()

## Conclusion

This notebook implements a fuzzy logic based Structural Health Assessment
system for an RCC beam using crack width and deflection as inputs. Triangular
membership functions were used throughout, and the final Structural Health
Index was calculated using the weighted average defuzzification method, as
required for this milestone.

The sample case and the 5 test cases show that as crack width and deflection
increase, the Structural Health Index decreases, correctly reflecting a
transition from Safe to Moderate to Critical condition. This matches the
expected engineering behaviour of an RCC beam.

As mentioned earlier, this model is only an academic demonstration of fuzzy
logic concepts. Real structural health assessment of an RCC beam must be
carried out through proper site inspection, material testing, structural
analysis and the judgement of a qualified structural engineer.

---
# Milestone 3
## Structural Health Assessment of an RCC Beam Using Fuzzy Logic

**Subject:** Fuzzy Logic and Genetic Programming
**GTU Subject Code:** BE05000461
**Milestone 3 / Phase 4, 5, 6 – Testing, Analysis and UI**

This part of the notebook continues directly from the Milestone-2 work above.
The same fuzzy controller (same inputs, same ranges, same membership functions
and same rules) is reused here. Nothing in the controller design is changed.
We only **test** it with more cases, **analyse** the results, and build a
simple **User Interface** on top of it.

# Phase 4 – Testing

In Milestone 2 (Step 17 above) we already tested the controller with 5
cases using the `evaluate_beam()` function. For Milestone 3 we extend this
to **10 test cases** so that all combinations of Low / Medium / High crack
width and deflection are covered, plus one extra realistic mixed case.

All readings below are picked only from the ranges already fixed in
Milestone 1 (Crack Width: 0–6 mm, Deflection: 0–25 mm). We reuse the same
`evaluate_beam()` function defined earlier — no new fuzzy logic is written.

In [ ]:
# 10 test cases covering Low/Medium/High combinations of both inputs
# (values are picked only from the existing 0-6 mm and 0-25 mm ranges)

milestone3_tests = [
    {"Test Case": "TC1  - Low crack + Low deflection",     "Crack Width (mm)": 0.3, "Deflection (mm)": 3},
    {"Test Case": "TC2  - Low crack + Medium deflection",  "Crack Width (mm)": 0.5, "Deflection (mm)": 12},
    {"Test Case": "TC3  - Low crack + High deflection",    "Crack Width (mm)": 0.4, "Deflection (mm)": 20},
    {"Test Case": "TC4  - Medium crack + Low deflection",  "Crack Width (mm)": 1.8, "Deflection (mm)": 4},
    {"Test Case": "TC5  - Medium crack + Medium deflection","Crack Width (mm)": 2.0, "Deflection (mm)": 12.5},
    {"Test Case": "TC6  - Medium crack + High deflection", "Crack Width (mm)": 1.9, "Deflection (mm)": 21},
    {"Test Case": "TC7  - High crack + Low deflection",    "Crack Width (mm)": 4.5, "Deflection (mm)": 3},
    {"Test Case": "TC8  - High crack + Medium deflection", "Crack Width (mm)": 5.0, "Deflection (mm)": 12},
    {"Test Case": "TC9  - High crack + High deflection",   "Crack Width (mm)": 5.5, "Deflection (mm)": 22},
    {"Test Case": "TC10 - Mixed / realistic case",         "Crack Width (mm)": 1.2, "Deflection (mm)": 9},
]

# run every case through the actual fuzzy controller (evaluate_beam, defined in Step 17)
for tc in milestone3_tests:
    idx, cond = evaluate_beam(tc["Crack Width (mm)"], tc["Deflection (mm)"])
    tc["Structural Health Index"] = idx
    tc["Health Condition"] = cond

milestone3_df = pd.DataFrame(milestone3_tests)
milestone3_df

### Testing Results Table

The table above (`milestone3_df`) shows all 10 test cases with the Structural
Health Index and Health Condition, both calculated directly by the fuzzy
controller. Below is a bar chart of the same results for a quick visual
comparison.

In [ ]:
plt.figure(figsize=(9, 5))
colors = ['green' if v > 65 else 'orange' if v > 35 else 'red'
          for v in milestone3_df["Structural Health Index"]]
plt.bar(milestone3_df["Test Case"], milestone3_df["Structural Health Index"], color=colors)
plt.ylabel("Structural Health Index")
plt.xticks(rotation=40, ha='right')
plt.axhline(65, color='green', linestyle=':', linewidth=1, label='Safe threshold (65)')
plt.axhline(35, color='red', linestyle=':', linewidth=1, label='Critical threshold (35)')
plt.title("Milestone 3 - Structural Health Index for 10 Test Cases")
plt.legend()
plt.tight_layout()
plt.show()

# Phase 5 – Analysis

This section analyses the 10 test cases from Phase 4.

### Controller Behaviour

**Effect of crack width:** Looking at TC1 → TC4 → TC7 (deflection kept low,
around 3–4 mm), the index drops from 85.0 (Safe) to 50.0 (Moderate) to 50.0
(Moderate) as crack width goes from 0.3 mm to 1.8 mm to 4.5 mm. So increasing
crack width alone is enough to pull the beam out of the Safe zone.

**Effect of deflection:** Looking at TC1 → TC2 → TC3 (crack width kept low,
around 0.3–0.5 mm), the index goes from 85.0 (Safe) to 85.0 (Safe) to 50.0
(Moderate) once deflection crosses into the High region (20 mm). So a beam
with a small crack can still slip into the Moderate zone once deflection
becomes large.

**Effect of both inputs being high:** TC9 (crack 5.5 mm, deflection 22 mm)
gives an index of 15.0 (Critical). This is the worst result among all 10
cases, which matches the expectation that a beam with both a wide crack and
large deflection is in the worst condition.

**Rule consistency check:** TC8 (high crack, medium deflection) and TC9
(high crack, high deflection) both land in Critical, matching Rules 8 and 9
in the rule base. TC1 and TC2 (low crack with low/medium deflection) both
land in Safe, matching Rules 1 and 2. So the controller output is behaving
exactly as the rule table defines - the mapping is consistent.

**Best vs worst cases:** TC1 and TC2 give the best health (85.0, Safe).
TC8 and TC9 give the worst health (15.0, Critical). TC10, the realistic
mixed case (crack 1.2 mm, deflection 9 mm), lands at 60.5 - just inside the
Moderate zone, which is a reasonable everyday result for a lightly cracked,
slightly sagging beam that still needs periodic monitoring.

### Strengths

- **Simple rule-based decisions:** Only 9 IF-THEN rules decide the whole
  output, so the logic is easy to trace back for any test case above.
- **Easy to interpret:** A single number (Structural Health Index) plus a
  label (Safe / Moderate / Critical) is far easier for a site engineer to
  read than two separate raw readings.
- **Handles gradual, uncertain conditions:** TC10 shows this well - the
  inputs are not clearly "low" or "high", yet the controller still gives a
  sensible middle-ground answer (60.5) instead of forcing a hard yes/no.
- **Easy to modify:** Since the rules are just 9 simple IF-THEN statements,
  changing a rule (e.g. making Rule 2 stricter) does not require touching
  the rest of the code.
- **Gives one clear numeric output:** The weighted-average index (0-100) is
  a single comparable score, which is what made the bar chart in Phase 4
  possible.

### Weaknesses

- **Assumed membership values:** The `a, b, c` values used for Low/Medium/
  High (e.g. crack High starting at 2.5 mm) are project-assumed numbers, not
  taken from a real IS-code table or lab data.
- **Manually defined rules:** All 9 rules were written using general
  engineering judgement, not derived from actual failure data of RCC beams.
- **Only two inputs used:** Only crack width and deflection are considered.
  TC7 (high crack, low deflection) still comes out as only "Moderate" and
  not "Critical", which shows the model cannot capture cases where crack
  width alone might already be dangerous for reasons outside these two
  inputs (e.g. crack depth, location, reinforcement corrosion).
- **Real beam behaviour is more complex:** Actual structural distress also
  depends on load history, material grade, environmental exposure, and
  reinforcement condition, none of which are modelled here.
- **Not a certification tool:** As noted throughout this notebook, the
  Structural Health Index from this controller is only an academic estimate
  and must never be used as a final structural safety certificate.

### Scope of Improvement

- Add more structural parameters as inputs (e.g. crack depth, span/depth
  ratio, vibration frequency) instead of only crack width and deflection.
- Replace the assumed membership function parameters with values calibrated
  from real field or lab data on RCC beams.
- Add more fuzzy rules (or finer linguistic levels, e.g. Very Low / Very
  High) so borderline cases like TC7 are classified more accurately.
- Connect the system to actual site sensors (crack-width gauges, LVDTs for
  deflection) so readings are logged automatically instead of typed in
  manually.
- Improve the UI (Phase 6) with a history/log of past readings so a beam can
  be tracked over time, not just a single snapshot.
- As a possible extension for this subject, Genetic Programming could be
  explored to automatically tune the membership function parameters instead
  of choosing them manually.

# Phase 6 – User Interface

### UI Implementation

We now build a simple, working UI directly inside the notebook using
`ipywidgets`. The UI uses the **same** `evaluate_beam()` function that was
already used in Phase 4 - no separate fuzzy controller is created. Run the
cell below, then enter a crack width and deflection value, and click the
**Assess Beam Health** button.

In [ ]:
!pip install ipywidgets -q

### UI Inputs and Output

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- UI header ---
title_html = widgets.HTML("<h2>RCC Beam Structural Health Assessment</h2>"
                           "<p><i>Fuzzy Logic Based Assessment</i></p>")

# --- UI inputs (same ranges as Milestone 1: crack 0-6 mm, deflection 0-25 mm) ---
crack_input = widgets.FloatText(value=1.0, description='Crack Width (mm):',
                                 style={'description_width': 'initial'})
deflection_input = widgets.FloatText(value=10.0, description='Deflection (mm):',
                                      style={'description_width': 'initial'})

assess_button = widgets.Button(description='Assess Beam Health', button_style='success')
output_box = widgets.Output()


def on_assess_clicked(b):
    with output_box:
        clear_output()
        cw = crack_input.value
        defl = deflection_input.value

        # basic range check, using the same ranges fixed in Milestone 1
        if not (0 <= cw <= 6) or not (0 <= defl <= 25):
            print("Please enter Crack Width between 0-6 mm and Deflection between 0-25 mm.")
            return

        # same fuzzy controller function used in Phase 4 testing
        idx, cond = evaluate_beam(cw, defl)

        print("Crack Width               :", cw, "mm")
        print("Deflection                 :", defl, "mm")
        print("Structural Health Index    :", idx)
        print("Health Condition           :", cond)


assess_button.on_click(on_assess_clicked)

display(title_html, crack_input, deflection_input, assess_button, output_box)

### Sample UI Result

Run the cell above, keep the default values (Crack Width = 1.0 mm,
Deflection = 10.0 mm) or enter your own readings within the fixed ranges,
then click **Assess Beam Health**. The output box will display the
Structural Health Index and Health Condition computed by the same fuzzy
controller used throughout this notebook. Take a screenshot of this
widget (with a filled-in result) for the Milestone-3 submission.

---
## Milestone 3 Summary

- **Phase 4 - Testing:** The same fuzzy controller from Milestone 2 was
  tested with 10 different crack width / deflection combinations, covering
  all Low/Medium/High pairs plus one realistic mixed case.
- **Phase 5 - Analysis:** The results were analysed for controller
  behaviour, strengths, weaknesses and scope of improvement, all tied back
  to the actual test case numbers.
- **Phase 6 - UI:** A simple `ipywidgets`-based interface was built on top
  of the same controller so that a user can enter readings and instantly
  see the Structural Health Index and Health Condition.

No part of the original fuzzy design (inputs, ranges, membership functions,
rules, or output scale) was changed for this milestone.